In [9]:
import pandas as pd
from tqdm.notebook import tqdm
from pprint import pprint
from tokenizers import Tokenizer
from tokenizers.trainers import BpeTrainer

In [10]:
file_export = '/mnt/Estudos/data/base_wiki_pt_cleaned_2.pq'

In [11]:
df = pd.read_parquet(file_export, engine='fastparquet')

FileNotFoundError: [Errno 2] No such file or directory: '/mnt/Estudos/data/base_wiki_pt_cleaned_2.pq'

In [ ]:
df.texto = df.texto.str.strip()
df.texto = df.texto.str.replace(r'\s+', ' ', regex=True)

In [ ]:
df.head()

,index,texto
0,1031610,O Opirus é um automóvel sedan de porte grande ...
1,1895765,Costa Macedo Giraldes Barba de Noronha e Brito...
2,649783,Eleutherodactylus cajamarcensis é uma espécie ...
3,1826843,Tricromatismo ou visão tricromática é a capaci...
4,1351999,Ode do grego antigo ōidē é um poema de estilo ...


In [ ]:
indexes = df.sample(frac=1, random_state=42).index.tolist()
perc_train, perc_eval, perc_test = 0.8, .015, 0.185
qtde_train, qtde_eval, qtde_test = int(len(indexes)*perc_train), int(len(indexes)*perc_eval), int(len(indexes)*perc_test)
indexes_train = indexes[:qtde_train]
indexes_eval = indexes[qtde_train: qtde_train+qtde_eval]
indexes_test = indexes[qtde_train+qtde_eval: -1]


df_train = df[df.index.isin(indexes_train)]
df_eval = df[df.index.isin(indexes_eval)]
df_test = df[df.index.isin(indexes_test)]

print(len(df_train), len(df_eval), len(df_test))

800000 15000 184999


In [ ]:
from tokenizers import Tokenizer
from tokenizers.normalizers import (Sequence, Lowercase, NFD, 
                                   StripAccents)
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.models import BPE
from tokenizers.decoders import ByteLevel as ByteLevelDecoder
from tokenizers.trainers import BpeTrainer

In [ ]:
from tokenizers.processors import TemplateProcessing
special_tokens=['[PAD]', '[BOS]', '[EOS]', '[UNK]']
temp_proc = TemplateProcessing(
    single="[BOS] $A [EOS]",
    pair=None,
    special_tokens=[
        ("[BOS]", special_tokens.index("[BOS]")),
        ("[EOS]", special_tokens.index("[EOS]")),
    ],
)

In [ ]:
tokenizer = Tokenizer(BPE())
tokenizer.pre_tokenizer = ByteLevel()
tokenizer.decoder = ByteLevelDecoder()
tokenizer.post_processor=temp_proc

In [ ]:
trainer = BpeTrainer(vocab_size=25000,special_tokens=special_tokens)
# tokenizer.train_from_iterator(df_train.texto.values, trainer=trainer)

In [ ]:
def batch_iterator(batch_size=1000):
    for i in range(0, len(df_train), batch_size):
        yield df_train.texto.iloc[i:i+batch_size].tolist()

tokenizer.train_from_iterator(
    batch_iterator(),
    trainer=trainer
)

KeyboardInterrupt: 

In [ ]:
def formar_paragrafos(text, n=200, p=0.75):
    paragrafos = text.split('.')
    novo_paragrafo = ''
    novos_tokens = []
    for x in range(len(paragrafos)):
        proposto = (novo_paragrafo + f'{paragrafos[x]}.').strip()
        tokens_proposto = novos_tokens + tokenizer.encode(f'{paragrafos[x]}.').ids
        len_proposto = len(tokens_proposto)
        if((len_proposto <= n*p) or (len_proposto > n*p and len_proposto <= n) ):
            novo_paragrafo = proposto
            novos_tokens = tokens_proposto
            continue
        else:
            break
    return novo_paragrafo, novos_tokens, len(novos_tokens)

In [ ]:
tokenizer = Tokenizer.from_file('artifacts/bpe_25000.json')

In [ ]:
tqdm.pandas(desc='Podando parágrafos')
n, p = 512, 0.75
df_train[['text_cut', 'tokens', 'len_text_cut']] = df_train.progress_apply(lambda x: formar_paragrafos(x['texto'], n, p), result_type='expand', axis=1)
df_eval[['text_cut', 'tokens', 'len_text_cut']] = df_eval.progress_apply(lambda x: formar_paragrafos(x['texto'], n, p), result_type='expand', axis=1)
df_test[['text_cut', 'tokens', 'len_text_cut']] = df_test.progress_apply(lambda x: formar_paragrafos(x['texto'], n, p), result_type='expand', axis=1)

Podando parágrafos:   0%|          | 0/800000 [00:00<?, ?it/s]

In [ ]:
tokenizer.save("artifacts/bpe_25000.json")

In [ ]:
tokenizer = Tokenizer.from_file("artifacts/bpe_25000.json")

In [ ]:
df_train.len_text_cut.describe()

count    800000.000000
mean        313.557351
std         169.525973
min           0.000000
25%         139.000000
50%         330.000000
75%         489.000000
max         512.000000
Name: len_text_cut, dtype: float64

In [ ]:
df_train_new = df_train[df_train.len_text_cut >= (n*p)]
df_eval_new = df_eval[df_eval.len_text_cut >= (n*p)]
df_test_new = df_test[df_test.len_text_cut >= (n*p)]

In [ ]:
len(df_train_new)

364510

In [ ]:
df_train_new = df_train_new.drop(columns=['texto'])
df_eval_new = df_eval_new.drop(columns=['texto'])
df_test_new = df_test_new.drop(columns=['texto'])

In [ ]:
df_train_new.to_parquet('data/train_wiki_cleaned_cutted.pq', index=False)
df_eval_new.to_parquet('data/eval_wiki_cleaned_cutted.pq', index=False)
df_test_new.to_parquet('data/test_wiki_cleaned_cutted.pq', index=False)

In [ ]:
df_test_new = pd.read_parquet('data/test_wiki_cleaned_cutted.pq')

In [ ]:
df_test_new[df_test_new.text_cut.str.contains('Lagrivea')].text_cut.values[0]

'.Lagrivea é um gênero fóssil de esquilo do Mioceno Médio da França. A única espécie, L. vireti, é conhecida por três mandíbulas maxilares inferiores e dois dentes isolados. Todos os restos provêm do preenchimento de fissura depósito fóssil formado quando uma fissura rochosa é preenchida com sedimento de La Grive L5, parte do complexo La Grive-Saint-Alban en em Saint-Alban-de-Roche, sudeste da França. Lagrivea era um grande esquilo florestal com incisivos inferiores planos e um quarto pré-molar inferior p4 grande e triangular. Cada um dos quatro dentes molares p4 e três molares, m1 a m3 apresenta uma bacia profunda no centro da coroa. O m3 tem formato aproximadamente retangular, mas arredondado na parte posterior. Embora m1 e m2 tenham duas raízes, o m3 possui três. Pierre Mein e Léonard Ginsburg descreveram Lagrivea vireti em 2002, em uma revisão das idades e faunas dos sítios fósseis do Mioceno de La Grive-Saint-Alban, no sudeste da França. Eles sugeriram que provavelmente era um esq

In [ ]:
tokenizer.decode(df_test_new[df_test_new.text_cut.str.contains('Lagrivea')].tokens.values[0])

' .Lagrivea é um gênero fóssil de esquilo do Mioceno Médio da França . A única espécie, L . vireti, é conhecida por três mandíbulas maxilares inferiores e dois dentes isolados . Todos os restos provêm do preenchimento de fissura depósito fóssil formado quando uma fissura rochosa é preenchida com sedimento de La Grive L5, parte do complexo La Grive-Saint-Alban en em Saint-Alban-de-Roche, sudeste da França . Lagrivea era um grande esquilo florestal com incisivos inferiores planos e um quarto pré-molar inferior p4 grande e triangular . Cada um dos quatro dentes molares p4 e três molares, m1 a m3 apresenta uma bacia profunda no centro da coroa . O m3 tem formato aproximadamente retangular, mas arredondado na parte posterior . Embora m1 e m2 tenham duas raízes, o m3 possui três . Pierre Mein e Léonard Ginsburg descreveram Lagrivea vireti em 2002, em uma revisão das idades e faunas dos sítios fósseis do Mioceno de La Grive-Saint-Alban, no sudeste da França . Eles sugeriram que provavelmente 

In [ ]:
df_train_new.len_text_cut.describe()

count    800000.000000
mean         37.460208
std          33.141298
min           0.000000
25%          23.000000
50%          32.000000
75%          44.000000
max         512.000000
Name: len_text_cut, dtype: float64

In [ ]:
df_train_new.tokens.values[1], df_train_new.len_text_cut.values[1]

([1,
  3784,
  14739,
  6543,
  1504,
  1124,
  1905,
  1234,
  1057,
  16650,
  53,
  13730,
  11473,
  1057,
  4146,
  1062,
  1109,
  118,
  1097,
  2160,
  1092,
  4344,
  17801,
  3386,
  1110,
  26,
  2],
 np.int64(27))

In [ ]:
len(df_train_new.tokens.values[484])

49

In [ ]:
df_train_new

,index,texto,text_cut,tokens,len_text_cut
780,895837,A Santa Casa de Misericórdia de Porto Alegre é...,A Santa Casa de Misericórdia de Porto Alegre é...,"[1, 23, 2560, 3669, 1057, 19060, 1057, 3096, 6...",447
852,990102,"Jorge Celso Gobbi OMM São Marcos, RS, 5 de dez...","Jorge Celso Gobbi OMM São Marcos, RS, 5 de dez...","[1, 4367, 17799, 29, 1262, 1188, 37, 19185, 15...",427
943,1629808,San Chirico Raparo é uma comuna italiana da re...,San Chirico Raparo é uma comuna italiana da re...,"[1, 2570, 2742, 2482, 1920, 8053, 118, 1111, 3...",386
979,226412,A artéria da retina central é um ramo da artér...,A artéria da retina central é um ramo da artér...,"[1, 23, 1070, 5735, 1063, 1073, 2938, 3462, 11...",397
1101,1874413,O Futebol de Areia do Club de Regatas Vasco da...,O Futebol de Areia do Club de Regatas Vasco da...,"[1, 37, 4262, 1057, 6183, 1072, 1058, 5718, 10...",460
...,...,...,...,...,...
998877,815421,Gobseck é um romance de Honoré de Balzac publi...,Gobseck é um romance de Honoré de Balzac publi...,"[1, 29, 1262, 1087, 1457, 118, 1097, 4791, 105...",407
999373,806222,"Gymnospermae do grego gimnós nu, spérma sement...","Gymnospermae do grego gimnós nu, spérma sement...","[1, 15076, 61, 1241, 1160, 1076, 53, 1058, 420...",465
999426,398812,SAI a princesa-viúva Napoléon Carlos Napoleão ...,SAI a princesa-viúva Napoléon Carlos Napoleão ...,"[1, 4222, 31, 49, 6629, 7, 12181, 1334, 24577,...",464
999543,1756342,Teano é uma comuna italiana da região da Campa...,Teano é uma comuna italiana da região da Campa...,"[1, 2045, 1349, 118, 1111, 3733, 6378, 1063, 1...",428


In [ ]:
df_train.text_cut.values[0], 

'O Opirus é um automóvel sedan de porte grande da Kia'

In [ ]:
pprint(df_train.texto.values[0])

('O Opirus é um automóvel sedan de porte grande da Kia. Como a primeira '
 'entrada da Kia para o grande carro mercado, o Opirus/Amanti tinha sido '
 'comercializado em um único nível de acabamento e apenas como um sedan. Ele '
 'compartilhou alguns componentes com o seu primo incorporado agora extinta, a '
 'Hyundai Grandeur XG, incluindo a sua 3.5 L V6. Para 2007, o Kia Opirus '
 'recebeu vários upgrades, incluindo a suspensão e revisão de estilo, e a '
 'adição do mesmo motor que o atual Hyundai Azera, desta vez sendo um 3,8 L '
 'V6. Nos EUA, o Opirus foi reconhecida como a mais atraente premium Midsize '
 'Car pela JD Power and Associates 2005 Desempenho Automotivo, Execução e '
 'Estudo de Layout. O Opirus 2007 superaram vários carros de luxo no Instituto '
 'de Seguros para a Segurança Rodoviária IIHS testes de colisão de impacto '
 'lateral, para ganhar a mais alta classificação do bem. A partir de 17 de '
 'dezembro de 2010, o site da Kia já não listou mais o Opirus como um mo

In [ ]:
tokenizer = Tokenizer.from_file('artifacts/bpe_25000.json')

In [14]:
frase = 'Lagrivea é um gênero fóssil de esquilo do Mioceno Médio da França . A única espécie, L . vireti, é conhecida por três mandíbulas maxilares inferiores e dois dentes isolados . Todos os restos provêm do preenchimento de fissura depósito fóssil formado quando uma fissura rochosa é preenchida com sedimento de La Grive L5, parte do complexo La Grive-Saint-Alban en em Saint-Alban-de-Roche, sudeste da França . Lagrivea era um grande esquilo florestal com incisivos inferiores planos e um quarto pré-molar inferior p4 grande e triangular . Cada um dos quatro dentes molares p4 e três molares, m1 a m3 apresenta uma bacia profunda no centro da coroa . O m3 tem formato aproximadamente retangular, mas arredondado na parte posterior . Embora m1 e m2 tenham duas raízes, o m3 possui três . Pierre Mein e Léonard Ginsburg descreveram Lagrivea vireti em 2002, em uma revisão das idades e faunas dos sítios fósseis do Mioceno de La Grive-Saint-Alban, no sudeste da França . Eles sugeriram que provavelmente era um esquilo florestal e relacionado a tribo Sciurini en . Lagrivea pertence à família dos esquilos Sciuridae, que aparece pela primeira vez no Eoceno Superior Priaboniano da América do Norte e no Oligoceno Inferior Rupeliano da Europa . O nome específico, vireti, homenageia Jean Viret por seu trabalho sobre os mamíferos de La Grive-Saint-Alban . Lagrivea é conhecido por três mandíbulas maxilares inferiores uma, o holótipo, com o quarto pré-molar p4 e todos os três molares m13 preservados; uma com o incisivo e o m2; e uma com o incisivo, p4, m1 e m2 além de um incisivo inferior isolado e um m2 isolado . Era grande para um esquilo, e pode ser distinguido dos esquilos fósseis Palaeosciurus, Aliveria e Ratufa obtusidens pelo seu tamanho maior'
ids = tokenizer.encode(frase).ids
ids

[1,
 17886,
 199,
 329,
 49,
 273,
 240,
 2443,
 17494,
 160,
 3762,
 1147,
 200,
 228,
 10965,
 1371,
 7616,
 208,
 1712,
 122,
 8,
 198,
 2397,
 1770,
 6,
 279,
 122,
 8,
 223,
 1219,
 1153,
 6,
 273,
 1885,
 259,
 871,
 16271,
 2385,
 21493,
 6879,
 11675,
 158,
 729,
 9718,
 18447,
 122,
 8,
 4610,
 276,
 7893,
 1093,
 834,
 200,
 13854,
 1414,
 160,
 184,
 636,
 409,
 18971,
 17494,
 4509,
 683,
 248,
 184,
 636,
 409,
 15528,
 1540,
 273,
 13854,
 351,
 197,
 19723,
 1414,
 160,
 962,
 9176,
 329,
 279,
 15,
 6,
 673,
 200,
 5022,
 962,
 9176,
 329,
 7,
 13052,
 1061,
 7,
 23,
 966,
 218,
 357,
 201,
 5579,
 7,
 23,
 966,
 218,
 7,
 303,
 7,
 40,
 22252,
 6,
 6908,
 208,
 1712,
 122,
 8,
 17886,
 199,
 329,
 49,
 514,
 240,
 798,
 3762,
 1147,
 20771,
 197,
 543,
 173,
 1282,
 11675,
 5199,
 158,
 240,
 3430,
 2224,
 7,
 61,
 11290,
 4951,
 162,
 14,
 798,
 158,
 22996,
 122,
 8,
 4751,
 240,
 320,
 1265,
 9718,
 203,
 12071,
 162,
 14,
 158,
 871,
 203,
 12071,
 6,
 203,
 11,
 1

In [19]:
pprint(tokenizer.decode(ids))

(' Lagrivea é um gênero fóssil de esquilo do Mioceno Médio da França . A única '
 'espécie, L . vireti, é conhecida por três mandíbulas maxilares inferiores e '
 'dois dentes isolados . Todos os restos provêm do preenchimento de fissura '
 'depósito fóssil formado quando uma fissura rochosa é preenchida com '
 'sedimento de La Grive L5, parte do complexo La Grive-Saint-Alban en em '
 'Saint-Alban-de-Roche, sudeste da França . Lagrivea era um grande esquilo '
 'florestal com incisivos inferiores planos e um quarto pré-molar inferior p4 '
 'grande e triangular . Cada um dos quatro dentes molares p4 e três molares, '
 'm1 a m3 apresenta uma bacia profunda no centro da coroa . O m3 tem formato '
 'aproximadamente retangular, mas arredondado na parte posterior . Embora m1 e '
 'm2 tenham duas raízes, o m3 possui três . Pierre Mein e Léonard Ginsburg '
 'descreveram Lagrivea vireti em 2002, em uma revisão das idades e faunas dos '
 'sítios fósseis do Mioceno de La Grive-Saint-Alban, no sudes

In [16]:
len(ids)

479

In [17]:
len(frase)

1748

In [18]:
len(ids) / len(frase)

0.2740274599542334